# MundaAI — Training Notebook
### AI-Powered Crop Risk Intelligence for Zimbabwe's Smallholder Farmers

**AI4I Challenge 2026 | Development Track**

This notebook trains the MundaAI crop risk classifier in 6 steps:
1. Install dependencies
2. Load & explore the dataset
3. Prepare features
4. CTGAN augmentation (fix class imbalance)
5. Train Random Forest classifier
6. Evaluate, visualise & export the model

⏱️ **Estimated runtime**: 20–30 minutes on Google Colab free tier

---

## Step 1 — Install Dependencies

> Run this cell first. It installs all required libraries. (~2 minutes)

In [ ]:
!pip install sdv imbalanced-learn scikit-learn pandas numpy matplotlib seaborn joblib -q
print('✅ All dependencies installed')

## Step 2 — Load & Explore the Dataset

> Upload `02_agriculture_climate_market_signals.csv` when prompted, or place it in your Colab files panel.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Load dataset ──────────────────────────────────────────────────────────────
# Option A: upload via Colab files panel, then use filename below
# Option B: if running locally, adjust path
try:
    df = pd.read_csv('02_agriculture_climate_market_signals.csv')
    print('Loaded from current directory')
except FileNotFoundError:
    from google.colab import files
    print('Please upload 02_agriculture_climate_market_signals.csv')
    uploaded = files.upload()
    df = pd.read_csv(list(uploaded.keys())[0])
    print('Loaded from upload')

print(f'\nDataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

In [ ]:
# ── Class imbalance — THE core problem CTGAN solves ──────────────────────────
print('Risk level distribution:')
dist = df['risk_level'].value_counts()
for level, count in dist.items():
    bar = '█' * int(count / 4)
    print(f'  {level:8s}: {count:3d} ({count/len(df)*100:.1f}%)  {bar}')

print(f'\n⚠️  HIGH RISK is only {dist.get("High", 0)/len(df)*100:.1f}% of data.')
print('   A standard model trained on this will miss High-risk cases.')
print('   CTGAN will generate synthetic High-risk samples to fix this.\n')

# Signal patterns by risk level
print('Average signals by risk level:')
summary = df.groupby('risk_level')[[
    'rainfall_mm', 'ndvi_proxy_0_1',
    'pest_incidents_reported', 'irrigation_coverage_pct'
]].mean().round(2)
print(summary)

In [ ]:
# ── Visualise class imbalance & signal patterns ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('MundaAI — Dataset Overview', fontsize=14, fontweight='bold')

# Class distribution
colors = {'Low': '#C0DD97', 'Medium': '#FAC775', 'High': '#F09595'}
dist.plot(kind='bar', ax=axes[0],
          color=[colors.get(k, 'gray') for k in dist.index])
axes[0].set_title('Class Distribution (Imbalance Problem)')
axes[0].set_xlabel('Risk Level')
axes[0].set_ylabel('Record count')
axes[0].tick_params(rotation=0)
for i, v in enumerate(dist):
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=10)

# Signal comparison
order = ['Low', 'Medium', 'High']
signals = ['rainfall_mm', 'ndvi_proxy_0_1', 'pest_incidents_reported']
signal_labels = ['Rainfall (mm)', 'NDVI (0-1)', 'Pest incidents']
sig_data = df.groupby('risk_level')[signals].mean().reindex(order)
sig_data.columns = signal_labels

# Normalise for comparison
sig_norm = (sig_data - sig_data.min()) / (sig_data.max() - sig_data.min())
sig_norm.plot(kind='bar', ax=axes[1], colormap='RdYlGn_r')
axes[1].set_title('Signal Patterns by Risk Level (normalised)')
axes[1].set_xlabel('Risk Level')
axes[1].tick_params(rotation=0)
axes[1].legend(loc='upper left', fontsize=8)

# Risk escalation over months
month_order = ['January', 'February', 'March', 'April', 'May', 'June']
month_risk = df.groupby(['month', 'risk_level']).size().unstack(fill_value=0)
available_months = [m for m in month_order if m in month_risk.index]
month_risk = month_risk.reindex(available_months)
month_risk[['Low', 'Medium', 'High']].plot(
    kind='bar', ax=axes[2], stacked=True,
    color=['#C0DD97', '#FAC775', '#F09595']
)
axes[2].set_title('Risk Escalation Jan → Jun')
axes[2].set_xlabel('Month')
axes[2].tick_params(rotation=45)

plt.tight_layout()
plt.savefig('dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: dataset_overview.png')

## Step 3 — Prepare Features

> Encode categorical columns (crop, province) into numbers the model can use.
> Split into training (80%) and test (20%) sets.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

df_model = df.copy()

# Label encode categorical columns
le_crop     = LabelEncoder()
le_province = LabelEncoder()
le_district = LabelEncoder()
le_risk     = LabelEncoder()

df_model['crop_encoded']     = le_crop.fit_transform(df_model['crop'])
df_model['province_encoded'] = le_province.fit_transform(df_model['province'])
df_model['district_encoded'] = le_district.fit_transform(df_model['district'])
df_model['risk_encoded']     = le_risk.fit_transform(df_model['risk_level'])

# Features the model learns from
FEATURES = [
    'rainfall_mm',
    'ndvi_proxy_0_1',
    'pest_incidents_reported',
    'irrigation_coverage_pct',
    'input_availability_score_0_100',
    'avg_farmgate_price_usd_per_tonne',
    'crop_encoded',
    'province_encoded'
]

X = df_model[FEATURES]
y = df_model['risk_encoded']
risk_classes = le_risk.classes_

print(f'Features ({len(FEATURES)}): {FEATURES}')
print(f'Target classes: {risk_classes}')
print(f'Class indices: Low={list(risk_classes).index("Low")}, Medium={list(risk_classes).index("Medium")}, High={list(risk_classes).index("High")}')

# Train/test split — stratify ensures proportional High-risk in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTraining set: {len(X_train)} records')
print(f'Test set: {len(X_test)} records')

## Step 4 — CTGAN Augmentation

> **The GAN step.** CTGAN learns the statistical patterns of High-risk records
> and generates synthetic ones to balance the training data.
> This is why the model can detect High-risk — it has seen enough examples.
>
> ⏱️ This cell takes ~10–15 minutes on Colab free tier.

In [ ]:
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

# ── Baseline model BEFORE augmentation (to show the problem) ─────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42)
rf_baseline.fit(X_train, y_train)
y_pred_base = rf_baseline.predict(X_test)

print('=== BASELINE MODEL — no augmentation ===')
print(classification_report(y_test, y_pred_base, target_names=risk_classes))
print('⚠️  Notice High risk recall — likely 0.00 or very low.')
print('   The model is blind to the most dangerous situations.\n')
print('-' * 55)

In [ ]:
# ── CTGAN: learn from real High-risk records ─────────────────────────────────
print('Training CTGAN on High-risk records...')
print('This teaches the GAN what High-risk conditions look like.\n')

# Extract only High-risk rows from training data
high_risk_idx = y_train == list(risk_classes).index('High')
X_high = X_train[high_risk_idx].copy()
X_high['risk_level'] = 'High'

print(f'Real High-risk training samples: {len(X_high)}')
print(f'CTGAN will generate synthetic samples to bring this to ~100\n')

# Set up metadata so CTGAN understands the data types
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(X_high)

# Train CTGAN
# epochs=300 gives good results on small datasets without taking too long
ctgan = CTGANSynthesizer(
    metadata,
    epochs=300,
    verbose=True,
    cuda=False  # Set to True if Colab GPU is available
)
ctgan.fit(X_high)
print('\n✅ CTGAN training complete')

In [ ]:
# ── Generate synthetic High-risk samples ─────────────────────────────────────
n_real_high = high_risk_idx.sum()
n_synthetic = max(85, 100 - n_real_high)  # Generate enough to balance classes

synthetic_df = ctgan.sample(num_rows=n_synthetic)
synthetic_df = synthetic_df[FEATURES].copy()  # Keep only model features

# Clip values to realistic ranges from the original dataset
for col in ['rainfall_mm', 'ndvi_proxy_0_1', 'pest_incidents_reported',
            'irrigation_coverage_pct', 'input_availability_score_0_100']:
    col_min = df[col].min()
    col_max = df[col].max()
    synthetic_df[col] = synthetic_df[col].clip(col_min, col_max)

# Assign High-risk label
high_encoded = list(risk_classes).index('High')
synthetic_y = pd.Series([high_encoded] * n_synthetic, name='risk_encoded')

print(f'✅ Generated {n_synthetic} synthetic High-risk records')
print(f'\nSample synthetic High-risk records:')
print(synthetic_df[['rainfall_mm', 'ndvi_proxy_0_1',
                     'pest_incidents_reported', 'irrigation_coverage_pct']].head(5).round(2))

## Step 5 — Train the Augmented Classifier

> Combine real training data + synthetic High-risk samples.
> Train Random Forest on the balanced dataset.

In [ ]:
# ── Combine real + synthetic data ────────────────────────────────────────────
X_train_aug = pd.concat([X_train, synthetic_df], ignore_index=True)
y_train_aug = pd.concat([
    pd.Series(y_train.values, name='risk_encoded'),
    synthetic_y
], ignore_index=True)

print('Augmented training set class distribution:')
for code_val, label in enumerate(risk_classes):
    count = (y_train_aug == code_val).sum()
    bar = '█' * int(count / 4)
    print(f'  {label:8s}: {count:3d}  {bar}')

# ── Train augmented Random Forest ─────────────────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight='balanced',  # Extra protection against residual imbalance
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_aug, y_train_aug)
y_pred_aug = rf_model.predict(X_test)

print('\n=== AUGMENTED MODEL — CTGAN + Random Forest ===')
print(classification_report(y_test, y_pred_aug, target_names=risk_classes))
print('✅ Notice the improvement in High risk recall vs baseline!')

## Step 6 — Evaluate, Visualise & Export

> Generate comparison charts, feature importance, and save the model.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('MundaAI — Model Results', fontsize=14, fontweight='bold')

# Confusion matrix — baseline
cm_base = confusion_matrix(y_test, y_pred_base)
disp_base = ConfusionMatrixDisplay(confusion_matrix=cm_base, display_labels=risk_classes)
disp_base.plot(ax=axes[0], colorbar=False, cmap='Reds')
axes[0].set_title('Baseline RF\n(misses High risk)')

# Confusion matrix — augmented
cm_aug = confusion_matrix(y_test, y_pred_aug)
disp_aug = ConfusionMatrixDisplay(confusion_matrix=cm_aug, display_labels=risk_classes)
disp_aug.plot(ax=axes[1], colorbar=False, cmap='Greens')
axes[1].set_title('CTGAN + RF (MundaAI)\n(detects High risk)')

# Feature importance
feat_imp = pd.Series(
    rf_model.feature_importances_,
    index=FEATURES
).sort_values(ascending=True)
feat_imp.plot(kind='barh', ax=axes[2], color='#1D9E75')
axes[2].set_title('Feature Importance\n(what drives risk predictions)')
axes[2].set_xlabel('Importance score')

plt.tight_layout()
plt.savefig('model_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: model_results.png')

In [ ]:
import joblib

# Save model and all encoders
joblib.dump(rf_model,     'mundaai_model.pkl')
joblib.dump(le_risk,      'le_risk.pkl')
joblib.dump(le_crop,      'le_crop.pkl')
joblib.dump(le_province,  'le_province.pkl')

print('✅ Model saved: mundaai_model.pkl')
print('✅ Encoders saved: le_risk.pkl, le_crop.pkl, le_province.pkl')
print('\nDownload these files from the Colab files panel (folder icon on left).')
print('Place them in the /model folder of the GitHub repo.')

In [ ]:
# ── Quick test prediction ─────────────────────────────────────────────────────
# Tests the exact Lupane groundnuts High-risk case from our dataset

def predict_risk(crop, province, rainfall, ndvi, pests, irrigation, input_avail, farmgate_price):
    """Predict crop risk for given conditions."""
    # Encode inputs
    crop_enc = le_crop.transform([crop])[0] if crop in le_crop.classes_ else 0
    prov_enc = le_province.transform([province])[0] if province in le_province.classes_ else 0

    features = pd.DataFrame([{
        'rainfall_mm': rainfall,
        'ndvi_proxy_0_1': ndvi,
        'pest_incidents_reported': pests,
        'irrigation_coverage_pct': irrigation,
        'input_availability_score_0_100': input_avail,
        'avg_farmgate_price_usd_per_tonne': farmgate_price,
        'crop_encoded': crop_enc,
        'province_encoded': prov_enc
    }])

    pred_encoded = rf_model.predict(features)[0]
    proba = rf_model.predict_proba(features)[0]
    risk_label = le_risk.inverse_transform([pred_encoded])[0]
    confidence = max(proba) * 100

    # Identify dominant stress factor
    importances = dict(zip(FEATURES, rf_model.feature_importances_))
    top_feature = max(importances, key=importances.get)
    factor_map = {
        'rainfall_mm': 'Rainfall deficit',
        'ndvi_proxy_0_1': 'Poor vegetation health (low NDVI)',
        'pest_incidents_reported': 'High pest pressure',
        'irrigation_coverage_pct': 'Limited irrigation coverage',
        'input_availability_score_0_100': 'Low input availability',
    }
    dominant = factor_map.get(top_feature, top_feature)

    return risk_label, confidence, dominant

# Test 1: Known High-risk case from dataset (Lupane, May 2026)
label, conf, factor = predict_risk(
    crop='Groundnuts', province='Matabeleland North',
    rainfall=7.1, ndvi=0.10, pests=10,
    irrigation=21.4, input_avail=41.3, farmgate_price=1308.8
)
print('=== TEST 1: Lupane Groundnuts (should be HIGH) ===')
print(f'Prediction: {label} ({conf:.1f}% confidence)')
print(f'Dominant factor: {factor}')

# Test 2: Low-risk conditions
label2, conf2, factor2 = predict_risk(
    crop='Maize', province='Mashonaland Central',
    rainfall=120.0, ndvi=0.65, pests=3,
    irrigation=55.0, input_avail=80.0, farmgate_price=450.0
)
print('\n=== TEST 2: Good conditions (should be LOW) ===')
print(f'Prediction: {label2} ({conf2:.1f}% confidence)')
print(f'Dominant factor: {factor2}')

print('\n✅ MundaAI model is working correctly!')
print('This model powers the /predict endpoint in the MundaAI API.')

## ✅ Training Complete

Download the following files from the **Colab files panel** (folder icon on the left sidebar):

- `mundaai_model.pkl` — the trained model
- `le_risk.pkl`, `le_crop.pkl`, `le_province.pkl` — the label encoders
- `dataset_overview.png` — EDA charts
- `model_results.png` — confusion matrices + feature importance

Place `mundaai_model.pkl` and the encoders in the `/model` folder of the GitHub repository.

---

**Next step**: The API in `/api/main.py` loads this model and serves predictions via REST endpoint.
See the main README for API setup instructions.